# Model Comparison & Rule-Based Baseline
**IE University · AI: Statistical Learning and Prediction**

This notebook benchmarks the full progression of approaches for document classification:

| Stage | Approach | Type |
|-------|----------|------|
| 0 | Keyword / Regex Rule-Based Classifier | Rule-based |
| 1 | Bag-of-Words + Logistic Regression | Classical ML (BoW baseline) |
| 2 | TF-IDF (word) + Logistic Regression | Classical ML |
| 3 | TF-IDF (word) + Naive Bayes | Classical ML |
| 4 | TF-IDF (word) + Random Forest | Classical ML |
| 5 | TF-IDF (word) + LinearSVC | Classical ML |
| 6 | TF-IDF (word + char n-grams) + LinearSVC | **Deployed model** |

For each model we report:
- Accuracy, Macro-F1, Macro-Precision, Macro-Recall
- Training time
- Per-class F1
- Confusion matrix
- 5-fold stratified cross-validation


In [ ]:
import sys, warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight

# ── style ───────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})

PALETTE = {
    'invoice':           '#60a5fa',
    'email':             '#34d399',
    'scientific_report': '#c084fc',
    'letter':            '#fbbf24',
}
MODEL_COLORS = [
    '#f87171', '#fb923c', '#facc15', '#4ade80',
    '#60a5fa', '#818cf8', '#e879f9',
]
LABELS = ['invoice', 'email', 'scientific_report', 'letter']

print('Libraries loaded.')

## 1 · Load Data

In [ ]:
train = pd.read_csv(ROOT / 'data/processed/train.csv').dropna(subset=['text', 'label'])
test  = pd.read_csv(ROOT / 'data/processed/test.csv').dropna(subset=['text', 'label'])

train = train[train['label'].isin(LABELS)]
test  = test[test['label'].isin(LABELS)]

X_train, y_train = train['text'].tolist(), train['label'].tolist()
X_test,  y_test  = test['text'].tolist(),  test['label'].tolist()

print(f'Train: {len(train):,} rows')
print(f'Test : {len(test):,} rows')
print()
print(train['label'].value_counts().to_string())

## 2 · Stage 0 — Rule-Based Regex Classifier

Before any machine learning, we establish a **rule-based baseline** using handcrafted
regular-expression patterns. Each document is scored against keyword patterns for every class;
the class with the highest match count wins.

This approach requires zero training data and zero learning — it encodes domain knowledge
about what each document type *looks like* textually.

In [ ]:
# ── Rule patterns (case-insensitive, anchored where helpful) ────────────────
RULE_PATTERNS = {
    'invoice': [
        r'\binvoice\b',
        r'\binv[-#]\w+',
        r'total\s*(?:amount|due|payable)',
        r'billing\s*date',
        r'due\s*date',
        r'\bvat\b',
        r'amount\s*due',
        r'bill\s*to',
        r'net\s*\d+\s*days',
        r'(?:subtotal|grand\s*total)',
        r'(?:USD|EUR|GBP)\s*[\d,]+\.\d{2}',
    ],
    'email': [
        r'from:\s*\S+@\S+',
        r'to:\s*\S+@\S+',
        r'subject:',
        r'cc:',
        r'bcc:',
        r'dear\s+\w+,',
        r'(?:best|kind)\s*regards',
        r'sincerely yours',
        r'reply\s*to',
        r'unsubscribe',
    ],
    'scientific_report': [
        r'\babstract\b',
        r'\bintroduction\b',
        r'\bmethodology\b',
        r'\bconclusion\b',
        r'et\s+al\.',
        r'p\s*[<>]\s*0\.0[0-9]',
        r'\breferences\b',
        r'\bfigure\s+\d',
        r'\btable\s+\d',
        r'(?:accuracy|precision|recall|f1)[\s\-]*score',
        r'neural\s*network',
        r'\bdataset\b',
    ],
    'letter': [
        r'dear\s+(?:sir|madam|hiring)',
        r'yours\s+(?:sincerely|faithfully)',
        r'i\s+am\s+writing\s+to',
        r'hereby\s+certif',
        r'tenancy\s+agreement',
        r'offer\s+(?:of\s+)?employment',
        r'notice\s+period',
        r'formal\s+complaint',
        r'landlord',
        r'hiring\s+manager',
    ],
}


def rule_based_classify(text: str) -> str:
    """Score text against each class's regex patterns; return argmax class."""
    t = text.lower()
    scores = {
        label: sum(1 for pat in patterns if re.search(pat, t))
        for label, patterns in RULE_PATTERNS.items()
    }
    best = max(scores, key=scores.get)
    # Tie-break: if all scores are 0, default to 'letter' (least distinctive)
    if scores[best] == 0:
        return 'letter'
    return best


t0 = time.time()
y_rule = [rule_based_classify(t) for t in X_test]
rule_time = time.time() - t0

rule_acc    = accuracy_score(y_test, y_rule)
rule_f1     = f1_score(y_test, y_rule, average='macro')
rule_prec   = precision_score(y_test, y_rule, average='macro', zero_division=0)
rule_recall = recall_score(y_test, y_rule, average='macro', zero_division=0)

print('Stage 0 · Rule-Based Classifier')
print(f'  Accuracy     : {rule_acc*100:.2f}%')
print(f'  Macro F1     : {rule_f1:.4f}')
print(f'  Macro Prec.  : {rule_prec:.4f}')
print(f'  Macro Recall : {rule_recall:.4f}')
print(f'  Inference time (test set): {rule_time:.3f}s')
print()
print(classification_report(y_test, y_rule, target_names=sorted(set(y_test))))

## 3 · Build Feature Matrices for ML Models

In [ ]:
print('Fitting vectorizers...')

# ── (a) Bag-of-Words (CountVectorizer, word unigrams) ────────────────────────
bow_vec = CountVectorizer(
    analyzer='word',
    min_df=2, max_df=0.95,
    max_features=60_000,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w\w+\b',
)
X_bow_train = bow_vec.fit_transform(X_train)
X_bow_test  = bow_vec.transform(X_test)
print(f'  BoW matrix : {X_bow_train.shape[0]:,} × {X_bow_train.shape[1]:,}')

# ── (b) TF-IDF word unigrams + bigrams ──────────────────────────────────────
tfidf_word = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2, max_df=0.95,
    max_features=80_000,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w\w+\b',
)
X_tfidf_train = tfidf_word.fit_transform(X_train)
X_tfidf_test  = tfidf_word.transform(X_test)
print(f'  TF-IDF word: {X_tfidf_train.shape[0]:,} × {X_tfidf_train.shape[1]:,}')

# ── (c) TF-IDF word + char (deployed model features) ────────────────────────
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=3, max_df=0.95,
    max_features=60_000,
    sublinear_tf=True,
    strip_accents='unicode',
)
X_char_train = tfidf_char.fit_transform(X_train)
X_char_test  = tfidf_char.transform(X_test)

X_dual_train = hstack([X_tfidf_train, X_char_train])
X_dual_test  = hstack([X_tfidf_test,  X_char_test])
print(f'  TF-IDF dual: {X_dual_train.shape[0]:,} × {X_dual_train.shape[1]:,}')

print('\nDone.')

## 4 · Train & Evaluate All Models

In [ ]:
def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, w))

cw = class_weights(y_train)

EXPERIMENTS = [
    # (name, classifier, X_train, X_test)
    ('Rule-Based (Regex)',
     None,  # handled separately
     None, None),

    ('BoW + Logistic Regression',
     LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42, solver='saga', n_jobs=-1),
     X_bow_train, X_bow_test),

    ('TF-IDF + Naive Bayes',
     MultinomialNB(alpha=0.1),
     X_tfidf_train, X_tfidf_test),

    ('TF-IDF + Logistic Regression',
     LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42, solver='saga', n_jobs=-1),
     X_tfidf_train, X_tfidf_test),

    ('TF-IDF + Random Forest',
     RandomForestClassifier(n_estimators=300, max_depth=None, class_weight='balanced', random_state=42, n_jobs=-1),
     X_tfidf_train, X_tfidf_test),

    ('TF-IDF + LinearSVC',
     LinearSVC(C=1.0, class_weight=cw, max_iter=2000, random_state=42),
     X_tfidf_train, X_tfidf_test),

    ('TF-IDF (word+char) + LinearSVC  ★ deployed',
     LinearSVC(C=1.0, class_weight=cw, max_iter=2000, random_state=42),
     X_dual_train, X_dual_test),
]

records   = []
per_class = {}  # name → per-class F1 dict
fitted    = {}  # name → fitted clf (for confusion matrices)

# Rule-based entry (pre-computed)
records.append({
    'Model':     'Rule-Based (Regex)',
    'Accuracy':  rule_acc,
    'F1 Macro':  rule_f1,
    'Precision': rule_prec,
    'Recall':    rule_recall,
    'Train (s)': 0.0,
})
per_class['Rule-Based (Regex)'] = {
    lbl: f1_score(y_test, y_rule, labels=[lbl], average='macro', zero_division=0)
    for lbl in LABELS
}

for name, clf, X_tr, X_te in EXPERIMENTS[1:]:  # skip rule-based
    print(f'Training: {name} ...', end=' ', flush=True)
    t0 = time.time()
    clf.fit(X_tr, y_train)
    train_t = round(time.time() - t0, 2)

    y_pred = clf.predict(X_te)
    acc    = accuracy_score(y_test, y_pred)
    f1_mac = f1_score(y_test, y_pred, average='macro')
    prec   = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec    = recall_score(y_test, y_pred, average='macro', zero_division=0)

    records.append({
        'Model':     name,
        'Accuracy':  acc,
        'F1 Macro':  f1_mac,
        'Precision': prec,
        'Recall':    rec,
        'Train (s)': train_t,
    })
    per_class[name] = {
        lbl: f1_score(y_test, y_pred, labels=[lbl], average='macro', zero_division=0)
        for lbl in LABELS
    }
    fitted[name] = (clf, X_te, y_pred)
    print(f'acc={acc*100:.2f}%  F1={f1_mac:.4f}  ({train_t}s)')

results_df = pd.DataFrame(records).set_index('Model')

# Format for display
display_df = results_df.copy()
display_df['Accuracy']  = display_df['Accuracy'].map('{:.4f}'.format)
display_df['F1 Macro']  = display_df['F1 Macro'].map('{:.4f}'.format)
display_df['Precision'] = display_df['Precision'].map('{:.4f}'.format)
display_df['Recall']    = display_df['Recall'].map('{:.4f}'.format)
display_df['Train (s)'] = display_df['Train (s)'].map('{:.2f}s'.format)

print()
display_df

## 5 · Comparison Charts

In [ ]:
# ── 5a  Accuracy + Macro F1 grouped bar ─────────────────────────────────────
names  = list(results_df.index)
x      = np.arange(len(names))
accs   = results_df['Accuracy'].values.astype(float)
f1s    = results_df['F1 Macro'].values.astype(float)
w      = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x - w/2, accs * 100, w,
               color=[MODEL_COLORS[i % len(MODEL_COLORS)] for i in range(len(names))],
               alpha=0.85, label='Accuracy (%)')
bars2 = ax.bar(x + w/2, f1s * 100, w,
               color=[MODEL_COLORS[i % len(MODEL_COLORS)] for i in range(len(names))],
               alpha=0.45, label='Macro F1 (%)')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
short_names = [
    'Rule-Based\n(Regex)',
    'BoW +\nLogReg',
    'TF-IDF +\nNaive Bayes',
    'TF-IDF +\nLogReg',
    'TF-IDF +\nRandom Forest',
    'TF-IDF +\nLinearSVC',
    'TF-IDF\n(word+char)\n+ LinearSVC ★',
]
ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylabel('Score (%)')
ax.set_title('Model Comparison — Accuracy & Macro F1 on Test Set', fontweight='bold', pad=14)
ax.set_ylim(0, 108)
ax.axhline(100, color='grey', lw=0.6, ls='--', alpha=0.4)
ax.legend(frameon=False)
# Highlight deployed model
ax.axvspan(len(names) - 1 - 0.5, len(names) - 0.5,
           alpha=0.06, color='gold', zorder=0)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/model_comparison_bar.png')

In [ ]:
# ── 5b  Per-class F1 heatmap ─────────────────────────────────────────────────
pc_df = pd.DataFrame(per_class, index=LABELS).T  # models × classes

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pc_df.values.astype(float), aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels([l.replace('_', ' ').title() for l in LABELS], fontsize=10)
ax.set_yticks(range(len(pc_df)))
short_yt = [
    'Rule-Based', 'BoW+LogReg', 'TFIDF+NB',
    'TFIDF+LogReg', 'TFIDF+RF', 'TFIDF+SVC', 'TFIDF(dual)+SVC ★',
]
ax.set_yticklabels(short_yt, fontsize=9)

for i in range(len(pc_df)):
    for j in range(len(LABELS)):
        val = pc_df.values[i, j]
        color = 'white' if val < 0.4 or val > 0.85 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='F1 Score')
ax.set_title('Per-Class F1 Score by Model', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/per_class_f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/per_class_f1_heatmap.png')

In [ ]:
# ── 5c  Training time vs accuracy scatter ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for i, (name, row) in enumerate(results_df.iterrows()):
    acc = float(row['Accuracy']) * 100
    t   = float(row['Train (s)'])
    col = MODEL_COLORS[i % len(MODEL_COLORS)]
    ax.scatter(t, acc, s=120, color=col, zorder=3, edgecolors='white', linewidths=0.6)
    label_text = short_names[i].replace('\n', ' ')
    ax.annotate(label_text, (t, acc),
                textcoords='offset points', xytext=(6, 3),
                fontsize=7.5, color=col)

ax.set_xlabel('Training Time (seconds)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs Training Time — Efficiency Frontier', fontweight='bold', pad=12)
ax.axhline(99, color='grey', lw=0.8, ls='--', alpha=0.5, label='99% threshold')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/accuracy_vs_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/accuracy_vs_time.png')

## 6 · Confusion Matrices

In [ ]:
# Rule-based confusion matrix
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

all_preds = [('Rule-Based (Regex)', y_rule)]
for name, (clf, X_te, y_pred) in fitted.items():
    all_preds.append((name, y_pred))

for ax, (name, y_pred) in zip(axes, all_preds):
    cm = confusion_matrix(y_test, y_pred, labels=LABELS)
    # Normalize by row (true labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)

    tick_labels = [l.replace('_', '\n') for l in LABELS]
    ax.set_xticks(range(len(LABELS)))
    ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(range(len(LABELS)))
    ax.set_yticklabels(tick_labels, fontsize=7)

    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            val = cm_norm[i, j]
            txt_color = 'white' if val > 0.6 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7.5, color=txt_color)

    short = name.replace('TF-IDF (word+char) + LinearSVC  ★ deployed', 'TF-IDF(dual)+SVC ★')
    ax.set_title(short, fontsize=8, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=7)
    ax.set_ylabel('True', fontsize=7)

# Hide extra axes if any
for ax in axes[len(all_preds):]:
    ax.set_visible(False)

plt.suptitle('Confusion Matrices — All Models (Row-Normalized)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/confusion_matrices_all.png')

## 7 · 5-Fold Stratified Cross-Validation

Cross-validation gives a more robust estimate of generalization performance than
a single train/test split, and guards against lucky or unlucky splits.

In [ ]:
CV_EXPERIMENTS = [
    ('BoW + Logistic Regression',
     LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42, solver='saga', n_jobs=-1),
     X_bow_train),

    ('TF-IDF + Naive Bayes',
     MultinomialNB(alpha=0.1),
     X_tfidf_train),

    ('TF-IDF + Logistic Regression',
     LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42, solver='saga', n_jobs=-1),
     X_tfidf_train),

    ('TF-IDF + Random Forest',
     RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
     X_tfidf_train),

    ('TF-IDF + LinearSVC',
     LinearSVC(C=1.0, class_weight=cw, max_iter=2000, random_state=42),
     X_tfidf_train),

    ('TF-IDF (word+char) + LinearSVC  ★ deployed',
     LinearSVC(C=1.0, class_weight=cw, max_iter=2000, random_state=42),
     X_dual_train),
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_records = []
for name, clf, X_tr in CV_EXPERIMENTS:
    print(f'  CV: {name} ...', end=' ', flush=True)
    t0 = time.time()
    scores = cross_val_score(clf, X_tr, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
    elapsed = round(time.time() - t0, 1)
    cv_records.append({
        'Model':       name,
        'CV F1 Mean':  round(scores.mean(), 4),
        'CV F1 Std':   round(scores.std(),  4),
        'Fold Scores': [round(s, 4) for s in scores],
        'Time (s)':    elapsed,
    })
    print(f'{scores.mean():.4f} ± {scores.std():.4f}  ({elapsed}s)')

cv_df = pd.DataFrame(cv_records).set_index('Model')
print()
cv_df[['CV F1 Mean', 'CV F1 Std', 'Time (s)']]

In [ ]:
# ── CV results chart ─────────────────────────────────────────────────────────
cv_names   = list(cv_df.index)
cv_means   = cv_df['CV F1 Mean'].values
cv_stds    = cv_df['CV F1 Std'].values

cv_short = [
    'BoW+LogReg', 'TFIDF+NB', 'TFIDF+LogReg',
    'TFIDF+RF', 'TFIDF+SVC', 'TFIDF(dual)+SVC ★',
]
x_cv = np.arange(len(cv_short))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(x_cv, cv_means * 100, color=MODEL_COLORS[1:],
              alpha=0.85, width=0.55)
ax.errorbar(x_cv, cv_means * 100, yerr=cv_stds * 100,
            fmt='none', color='#1e293b', capsize=5, linewidth=1.5)

for bar, mean, std in zip(bars, cv_means, cv_stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + cv_stds.max()*100 + 0.5,
            f'{mean*100:.2f}\n±{std*100:.2f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x_cv)
ax.set_xticklabels(cv_short, fontsize=9)
ax.set_ylabel('Macro F1 (%)')
ax.set_title('5-Fold Stratified CV — Macro F1 (mean ± std)', fontweight='bold', pad=12)
ax.set_ylim(0, 115)
ax.axhline(99, color='grey', lw=0.8, ls='--', alpha=0.5)
# Highlight deployed
ax.axvspan(len(cv_short) - 1 - 0.4, len(cv_short) - 0.6,
           alpha=0.07, color='gold', zorder=0)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/cv_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/cv_results.png')

## 8 · Hyperparameter Sensitivity Analysis — LinearSVC

In [ ]:
# Sweep regularization parameter C for the deployed (dual TF-IDF) + LinearSVC
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
C_scores = []

for C in C_values:
    clf_c = LinearSVC(C=C, class_weight=cw, max_iter=2000, random_state=42)
    clf_c.fit(X_dual_train, y_train)
    y_p = clf_c.predict(X_dual_test)
    C_scores.append(f1_score(y_test, y_p, average='macro'))
    print(f'  C={C:<6}  F1={C_scores[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(C_values, [s*100 for s in C_scores],
            marker='o', color='#818cf8', linewidth=2, markersize=7)
best_idx = int(np.argmax(C_scores))
ax.scatter([C_values[best_idx]], [C_scores[best_idx]*100],
           s=140, color='gold', zorder=5, edgecolors='#1e293b', linewidths=1, label=f'Best C={C_values[best_idx]}')
ax.set_xlabel('Regularization parameter C (log scale)')
ax.set_ylabel('Macro F1 (%)')
ax.set_title('LinearSVC Hyperparameter Sensitivity (C sweep)', fontweight='bold', pad=12)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/hyperparam_C_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best C={C_values[best_idx]}  →  F1={max(C_scores)*100:.4f}%')

## 9 · Feature Analysis — Top TF-IDF Discriminating Terms

In [ ]:
# Use the deployed LinearSVC (dual TF-IDF)
clf_deployed = LinearSVC(C=1.0, class_weight=cw, max_iter=2000, random_state=42)
clf_deployed.fit(X_dual_train, y_train)

word_features = tfidf_word.get_feature_names_out()
char_features = tfidf_char.get_feature_names_out()
all_features  = np.concatenate([word_features, char_features])

N_TOP = 15
fig, axes = plt.subplots(1, 4, figsize=(18, 6))

for ax, label in zip(axes, clf_deployed.classes_):
    idx   = list(clf_deployed.classes_).index(label)
    coefs = clf_deployed.coef_[idx]

    # Word features only (more interpretable)
    n_word = len(word_features)
    word_coefs = coefs[:n_word]
    top_idx    = np.argsort(word_coefs)[-N_TOP:][::-1]
    top_feats  = word_features[top_idx]
    top_vals   = word_coefs[top_idx]

    color = list(PALETTE.values())[list(PALETTE.keys()).index(label)] if label in PALETTE else '#94a3b8'
    bars  = ax.barh(range(N_TOP), top_vals[::-1],
                    color=color, alpha=0.8)
    ax.set_yticks(range(N_TOP))
    ax.set_yticklabels(top_feats[::-1], fontsize=8)
    ax.set_title(label.replace('_', ' ').title(), fontweight='bold', color=color)
    ax.set_xlabel('SVC weight')
    ax.spines['left'].set_visible(False)

plt.suptitle('Top 15 Discriminating TF-IDF Terms per Class (Word n-grams)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/top_features_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/top_features_per_class.png')

## 10 · Rule-Based Classifier — Deep Dive

We examine where the rule-based approach fails to understand its limitations
and justify why statistical ML is necessary.

In [ ]:
errors_rule = test.copy()
errors_rule['predicted'] = y_rule
errors_rule = errors_rule[errors_rule['label'] != errors_rule['predicted']]

print(f'Total rule-based errors: {len(errors_rule)} / {len(test)} ({len(errors_rule)/len(test)*100:.1f}%)')
print()
print('Error breakdown (true → predicted):')
breakdown = errors_rule.groupby(['label', 'predicted']).size().reset_index(name='count')
print(breakdown.to_string(index=False))
print()

# ── Show a few examples where rule-based fails but LinearSVC succeeds ────────
deployed_preds = clf_deployed.predict(X_dual_test)
rule_wrong_ml_right = [
    i for i, (yt, yr, ym) in enumerate(zip(y_test, y_rule, deployed_preds))
    if yr != yt and ym == yt
]
print(f'Cases where rule-based fails but LinearSVC is correct: {len(rule_wrong_ml_right)}')
print()
for i in rule_wrong_ml_right[:3]:
    print(f'  True: {y_test[i]}  |  Rule: {y_rule[i]}  |  LinearSVC: {deployed_preds[i]}')
    print(f'  Preview: {X_test[i][:200].replace(chr(10)," ")}...')
    print()

## 11 · Summary Table & Conclusion

In [ ]:
# Merge test results with CV results
summary = results_df[['Accuracy', 'F1 Macro', 'Train (s)']].copy()
summary['Accuracy %'] = (summary['Accuracy'].astype(float) * 100).round(2)
summary['F1 Macro %'] = (summary['F1 Macro'].astype(float) * 100).round(2)

cv_means_dict = {r['Model']: r['CV F1 Mean'] for r in cv_records}
cv_stds_dict  = {r['Model']: r['CV F1 Std']  for r in cv_records}

summary['CV F1 (5-fold)'] = [
    f"{cv_means_dict.get(n, float('nan'))*100:.2f} ± {cv_stds_dict.get(n, float('nan'))*100:.2f}"
    if n in cv_means_dict else 'N/A'
    for n in summary.index
]

print('=== FINAL MODEL COMPARISON SUMMARY ===')
print(summary[['Accuracy %', 'F1 Macro %', 'CV F1 (5-fold)', 'Train (s)']].to_string())

print()
print('★ Deployed model: TF-IDF (word+char) + LinearSVC')
print()
print('Conclusion:')
print('  • Rule-based regex achieves reasonable accuracy on clear-cut cases')
print('    but fails on ambiguous or mixed-format documents.')
print('  • Adding TF-IDF features dramatically improves over BoW.')
print('  • LinearSVC outperforms Random Forest and Logistic Regression on this')
print('    high-dimensional sparse feature space (theory: large margin, no')
print('    overfitting on sparse inputs).')
print('  • Adding character n-grams on top of word n-grams captures format')
print('    cues (punctuation patterns, digit sequences) that word tokens miss,')
print('    pushing the deployed model to near-perfect performance.')
print('  • Cross-validation confirms the result is robust (low std across folds).')

In [ ]:
# ── Final radar / spider chart comparing top-3 models ───────────────────────
from matplotlib.patches import FancyArrowPatch

categories = ['Accuracy', 'Macro F1', 'Macro Prec.', 'Macro Recall', 'Speed\n(inv. train time)']
N = len(categories)

def get_radar_vals(model_name):
    row = results_df.loc[model_name]
    acc   = float(row['Accuracy'])
    f1    = float(row['F1 Macro'])
    prec  = float(row['Precision'])
    rec   = float(row['Recall'])
    t     = float(row['Train (s)'])
    # Normalize speed: faster = higher score. Use 1/(1+t) scaled to [0,1]
    # Reference: max train time in the set
    max_t  = max(results_df['Train (s)'].astype(float))
    speed  = 1 - (t / (max_t + 1))
    return [acc, f1, prec, rec, speed]

radar_models = [
    ('Rule-Based (Regex)',                          '#f87171'),
    ('TF-IDF + Random Forest',                      '#4ade80'),
    ('TF-IDF (word+char) + LinearSVC  ★ deployed', '#818cf8'),
]

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the ring

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0f172a')

for (name, color) in radar_models:
    vals = get_radar_vals(name) + [get_radar_vals(name)[0]]
    display_label = name.replace('TF-IDF (word+char) + LinearSVC  ★ deployed', 'TFIDF(dual)+SVC ★')
    ax.plot(angles, vals, color=color, linewidth=2, label=display_label)
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_thetagrids(np.degrees(angles[:-1]), categories, fontsize=10, color='#e2e8f0')
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=7, color='#94a3b8')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values():
    spine.set_color('#334155')
ax.grid(color='#334155', linestyle='-', linewidth=0.6)

legend = ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
                   frameon=False, fontsize=9)
for text in legend.get_texts():
    text.set_color('#e2e8f0')

ax.set_title('Model Capability Radar', color='#e2e8f0', fontsize=13,
             fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(ROOT / 'notebooks/radar_chart.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved → notebooks/radar_chart.png')